# Tahoe-100M Data Loading and DataLoader Pipeline

This notebook streams Tahoe-100M from HuggingFace, organizes records as **cell line -> perturbation -> expression matrix**, and prepares PyTorch DataLoaders using an **80/10/10 perturbation-level split** for predicting unseen perturbations on known cell lines.

In [1]:
from collections import defaultdict
import time

import numpy as np
import pandas as pd
import anndata as ad
import requests
import torch
from datasets import load_dataset
from scipy.sparse import csr_matrix
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

In [2]:
DATASET_NAME = "vevotx/Tahoe-100M"
SAMPLE_SIZE = 200_000  # Set to None to stream the full dataset (very large).
REPORT_EVERY = 25_000

ENSEMBL_LOOKUP_BATCH_SIZE = 500
ENSEMBL_LOOKUP_PAUSE_SECONDS = 0.05
ENSEMBL_LOOKUP_TIMEOUT_SECONDS = 30


def fetch_ensembl_biotypes(
    ensembl_ids,
    batch_size=ENSEMBL_LOOKUP_BATCH_SIZE,
    pause_seconds=ENSEMBL_LOOKUP_PAUSE_SECONDS,
    timeout_seconds=ENSEMBL_LOOKUP_TIMEOUT_SECONDS,
):
    unique_ids = sorted(
        {
            str(gene_id).split(".")[0]
            for gene_id in ensembl_ids
            if isinstance(gene_id, str) and gene_id
        }
    )

    lookup_url = "https://rest.ensembl.org/lookup/id"
    headers = {"Content-Type": "application/json", "Accept": "application/json"}

    biotype_by_id = {}
    failed_batches = 0

    for batch_start in range(0, len(unique_ids), batch_size):
        batch_ids = unique_ids[batch_start : batch_start + batch_size]
        payload = {"ids": batch_ids}

        try:
            response = requests.post(
                lookup_url,
                headers=headers,
                json=payload,
                timeout=timeout_seconds,
            )
            response.raise_for_status()
            response_payload = response.json()

            for gene_id in batch_ids:
                annotation = response_payload.get(gene_id)
                biotype_by_id[gene_id] = (
                    annotation.get("biotype") if isinstance(annotation, dict) else None
                )
        except requests.RequestException:
            failed_batches += 1
            for gene_id in batch_ids:
                biotype_by_id.setdefault(gene_id, None)

        if pause_seconds:
            time.sleep(pause_seconds)

    return biotype_by_id, failed_batches


def build_filtered_gene_vocab(gene_metadata):
    gene_df = gene_metadata.to_pandas()
    required_columns = {"token_id", "ensembl_id"}
    missing_columns = required_columns - set(gene_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required gene metadata columns: {sorted(missing_columns)}")

    gene_df = gene_df.loc[:, ["token_id", "ensembl_id"]].dropna()
    initial_genes = int(len(gene_df))

    gene_df = gene_df.drop_duplicates(subset="ensembl_id", keep="first").copy()
    after_dedup_genes = int(len(gene_df))

    gene_df["ensembl_core"] = gene_df["ensembl_id"].astype(str).str.split(".").str[0]

    biotype_by_id, failed_batches = fetch_ensembl_biotypes(gene_df["ensembl_core"].tolist())
    gene_df["biotype"] = gene_df["ensembl_core"].map(biotype_by_id)

    resolved_biotypes = int(gene_df["biotype"].notna().sum())
    if resolved_biotypes == 0:
        raise RuntimeError(
            "External Ensembl lookup returned no biotype annotations. "
            "Cannot remove pseudogenes without external annotation."
        )

    pseudogene_mask = gene_df["biotype"].fillna("").str.contains("pseudogene", case=False)
    pseudogenes_removed = int(pseudogene_mask.sum())
    gene_df = gene_df.loc[~pseudogene_mask].copy()
    after_pseudogene_genes = int(len(gene_df))

    duplicated_token_ids = int(gene_df["token_id"].duplicated().sum())
    if duplicated_token_ids:
        gene_df = gene_df.drop_duplicates(subset="token_id", keep="first").copy()

    gene_vocab = {
        int(token_id): ensembl_id
        for token_id, ensembl_id in zip(gene_df["token_id"], gene_df["ensembl_id"])
    }

    stats = {
        "initial_genes": initial_genes,
        "after_dedup_genes": after_dedup_genes,
        "pseudogenes_removed": pseudogenes_removed,
        "after_pseudogene_genes": after_pseudogene_genes,
        "resolved_biotypes": resolved_biotypes,
        "unresolved_biotypes": int(after_dedup_genes - resolved_biotypes),
        "failed_lookup_batches": failed_batches,
        "duplicated_token_ids_removed": duplicated_token_ids,
    }

    return gene_vocab, stats


def build_cellline_perturbation_index(streaming_ds, gene_vocab, sample_size=None, report_every=None):
    sorted_vocab_items = sorted(gene_vocab.items())
    token_ids, gene_names = zip(*sorted_vocab_items)
    token_id_to_col_idx = {token_id: idx for idx, token_id in enumerate(token_ids)}

    grouped_buffers = defaultdict(
        lambda: {"data": [], "indices": [], "indptr": [0], "obs": []}
    )

    for i, record in enumerate(streaming_ds):
        if sample_size is not None and i >= sample_size:
            break

        genes = record["genes"]
        expressions = record["expressions"]

        # The tutorial notebook drops a leading sentinel value when expression starts negative.
        if expressions and expressions[0] < 0:
            genes = genes[1:]
            expressions = expressions[1:]

        row_indices = []
        row_values = []
        for gene_token, expr_value in zip(genes, expressions):
            col_idx = token_id_to_col_idx.get(gene_token)
            if col_idx is not None:
                row_indices.append(col_idx)
                row_values.append(float(expr_value))

        key = (record["cell_line_id"], record["drug"])
        buffer = grouped_buffers[key]
        buffer["indices"].extend(row_indices)
        buffer["data"].extend(row_values)
        buffer["indptr"].append(len(buffer["data"]))
        buffer["obs"].append(
            {k: v for k, v in record.items() if k not in ("genes", "expressions")}
        )

        if report_every and (i + 1) % report_every == 0:
            print(f"Streamed {i + 1:,} cells. Current groups: {len(grouped_buffers):,}")

    data_by_cell_and_drug = defaultdict(dict)
    var_index = pd.Index(gene_names, name="ensembl_id")

    for (cell_line_id, perturbation), buffer in grouped_buffers.items():
        x_matrix = csr_matrix(
            (
                np.asarray(buffer["data"], dtype=np.float32),
                np.asarray(buffer["indices"], dtype=np.int32),
                np.asarray(buffer["indptr"], dtype=np.int64),
            ),
            shape=(len(buffer["obs"]), len(gene_names)),
            dtype=np.float32,
        )
        x_matrix.sum_duplicates()
        x_matrix.eliminate_zeros()

        obs_df = pd.DataFrame(buffer["obs"])
        pair_adata = ad.AnnData(X=x_matrix, obs=obs_df)
        pair_adata.var.index = var_index
        data_by_cell_and_drug[cell_line_id][perturbation] = pair_adata

    return data_by_cell_and_drug, list(gene_names)


def filter_to_globally_nonzero_genes(data_by_cell_and_drug, gene_names):
    if not gene_names:
        return data_by_cell_and_drug, gene_names, {
            "total_cells": 0,
            "input_genes": 0,
            "genes_with_any_zero": 0,
            "kept_genes": 0,
        }

    total_cells = 0
    nonzero_counts = np.zeros(len(gene_names), dtype=np.int64)

    for perturbation_map in data_by_cell_and_drug.values():
        for pair_adata in perturbation_map.values():
            matrix = pair_adata.X.tocsr() if hasattr(pair_adata.X, "tocsr") else csr_matrix(pair_adata.X)
            matrix.sum_duplicates()
            matrix.eliminate_zeros()

            binary_matrix = matrix.copy()
            binary_matrix.data = np.ones(binary_matrix.nnz, dtype=np.int8)

            nonzero_counts += np.asarray(binary_matrix.sum(axis=0)).ravel().astype(np.int64, copy=False)
            total_cells += matrix.shape[0]

    keep_mask = nonzero_counts == total_cells
    kept_gene_names = [gene_name for gene_name, keep in zip(gene_names, keep_mask) if keep]

    for cell_line_id, perturbation_map in data_by_cell_and_drug.items():
        for perturbation, pair_adata in perturbation_map.items():
            filtered_pair_adata = pair_adata[:, keep_mask].copy()
            filtered_pair_adata.var.index = pd.Index(kept_gene_names, name="ensembl_id")
            perturbation_map[perturbation] = filtered_pair_adata

    stats = {
        "total_cells": int(total_cells),
        "input_genes": int(len(gene_names)),
        "genes_with_any_zero": int((~keep_mask).sum()),
        "kept_genes": int(keep_mask.sum()),
    }

    return data_by_cell_and_drug, kept_gene_names, stats


tahoe_100m_stream = load_dataset(DATASET_NAME, streaming=True, split="train")
gene_metadata = load_dataset(DATASET_NAME, name="gene_metadata", split="train")
gene_vocab, gene_filter_stats = build_filtered_gene_vocab(gene_metadata)

data_by_cell_and_drug, gene_names = build_cellline_perturbation_index(
    tahoe_100m_stream,
    gene_vocab,
    sample_size=SAMPLE_SIZE,
    report_every=REPORT_EVERY,
)

data_by_cell_and_drug, gene_names, nonzero_stats = filter_to_globally_nonzero_genes(
    data_by_cell_and_drug,
    gene_names,
)

print(f"Gene features before filtering: {gene_filter_stats['initial_genes']:,}")
print(f"After duplicate drop: {gene_filter_stats['after_dedup_genes']:,}")
print(f"After pseudogene drop: {gene_filter_stats['after_pseudogene_genes']:,}")
print(f"After any-zero drop: {nonzero_stats['kept_genes']:,}")
print(f"Cell lines observed: {len(data_by_cell_and_drug):,}")

if gene_filter_stats["failed_lookup_batches"]:
    print(
        f"Warning: {gene_filter_stats['failed_lookup_batches']} Ensembl lookup batches failed. "
        "Some pseudogenes may remain if those lookups were unresolved."
    )
if gene_filter_stats["unresolved_biotypes"]:
    print(
        f"Warning: {gene_filter_stats['unresolved_biotypes']:,} genes have unresolved biotypes "
        "after external lookup."
    )
if nonzero_stats["kept_genes"] == 0:
    print("Warning: no genes remain after removing columns that contain any zero.")
elif nonzero_stats["kept_genes"] < 100:
    print("Warning: fewer than 100 genes remain after strict nonzero filtering.")

if len(gene_names) != len(set(gene_names)):
    raise ValueError("Duplicate gene names remain after filtering.")

all_entries = 0
total_nonzero = 0
for perturbation_map in data_by_cell_and_drug.values():
    for pair_adata in perturbation_map.values():
        all_entries += int(pair_adata.n_obs * pair_adata.n_vars)
        total_nonzero += int(pair_adata.X.nnz)

if all_entries and total_nonzero != all_entries:
    raise ValueError("Filtered matrices still contain zero entries.")

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Streamed 25,000 cells. Current groups: 2,665
Streamed 50,000 cells. Current groups: 4,393
Streamed 75,000 cells. Current groups: 4,501
Streamed 100,000 cells. Current groups: 4,559
Streamed 125,000 cells. Current groups: 4,596
Streamed 150,000 cells. Current groups: 4,619
Streamed 175,000 cells. Current groups: 4,646
Streamed 200,000 cells. Current groups: 4,656


/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/Users/aniruddh/miniforge3/envs/DeepLearningClass/lib/p

Gene features before filtering: 62,710
After duplicate drop: 62,710
After pseudogene drop: 47,528
After any-zero drop: 0
Cell lines observed: 50


In [3]:
def summarize_hierarchy(data_by_cell_and_drug):
    rows = []
    for cell_line_id, perturbation_map in data_by_cell_and_drug.items():
        for perturbation, pair_adata in perturbation_map.items():
            rows.append(
                {
                    "cell_line_id": cell_line_id,
                    "perturbation": perturbation,
                    "n_cells": int(pair_adata.n_obs),
                    "n_genes": int(pair_adata.n_vars),
                }
            )
    return pd.DataFrame(rows)


def expression_df_for_pair(data_by_cell_and_drug, cell_line_id, perturbation):
    pair_adata = data_by_cell_and_drug[cell_line_id][perturbation]
    if "BARCODE_SUB_LIB_ID" in pair_adata.obs.columns:
        row_index = pair_adata.obs["BARCODE_SUB_LIB_ID"].astype(str)
    else:
        row_index = pair_adata.obs.index.astype(str)

    expr_df = pd.DataFrame.sparse.from_spmatrix(
        pair_adata.X,
        index=row_index,
        columns=pair_adata.var_names,
    )
    return expr_df


pair_summary_df = summarize_hierarchy(data_by_cell_and_drug)

if pair_summary_df.empty:
    print("No records were loaded. Increase SAMPLE_SIZE or check dataset access.")
else:
    cell_line_summary_df = (
        pair_summary_df.groupby("cell_line_id", as_index=False)
        .agg(
            n_perturbations=("perturbation", "nunique"),
            n_cells=("n_cells", "sum"),
        )
        .sort_values(["n_perturbations", "n_cells"], ascending=False)
    )

    print(f"Cell lines indexed: {pair_summary_df['cell_line_id'].nunique():,}")
    print(f"Unique perturbations observed: {pair_summary_df['perturbation'].nunique():,}")
    print(f"Cell-line/perturbation groups: {len(pair_summary_df):,}")

    display(cell_line_summary_df.head(10))
    display(pair_summary_df.sort_values("n_cells", ascending=False).head(10))

    example_cell_line, example_perturbation = pair_summary_df.loc[
        0, ["cell_line_id", "perturbation"]
    ]
    example_expr_df = expression_df_for_pair(
        data_by_cell_and_drug,
        example_cell_line,
        example_perturbation,
    )

    print(
        f"Example pair ({example_cell_line}, {example_perturbation}) expression matrix shape: "
        f"{example_expr_df.shape}"
    )
    display(example_expr_df.iloc[:3, :10])

Cell lines indexed: 50
Unique perturbations observed: 95
Cell-line/perturbation groups: 4,656


,cell_line_id,n_perturbations,n_cells
22,CVCL_0546,95,11462
19,CVCL_0459,95,9651
20,CVCL_0480,95,7847
31,CVCL_1285,95,6979
4,CVCL_0131,95,6620
0,CVCL_0023,95,6460
17,CVCL_0399,95,6376
35,CVCL_1517,95,6313
24,CVCL_1056,95,6282
13,CVCL_0359,95,6104


,cell_line_id,perturbation,n_cells,n_genes
189,CVCL_0546,DMSO_TF,243,0
187,CVCL_0546,Clonidine (hydrochloride),234,0
99,CVCL_0546,Retinoic acid,208,0
1123,CVCL_0459,Goserelin (acetate),207,0
103,CVCL_0546,Riluzole hydrochloride,205,0
102,CVCL_0546,Pasireotide (acetate),205,0
186,CVCL_0546,Anethole trithione,200,0
1070,CVCL_0459,Quinestrol,198,0
1136,CVCL_0459,Clonidine (hydrochloride),197,0
94,CVCL_0480,DMSO_TF,194,0


Example pair (CVCL_0480, 8-Hydroxyquinoline) expression matrix shape: (115, 0)


ensembl_id
BARCODE_SUB_LIB_ID
01_001_052-lib_1105
01_011_106-lib_1105
01_023_046-lib_1105


In [4]:
sample_metadata = load_dataset(DATASET_NAME, name="sample_metadata", split="train")
all_perturbations = sorted(set(sample_metadata["drug"]))

rng = np.random.default_rng(42)
shuffled_perturbations = np.array(all_perturbations, dtype=object)
rng.shuffle(shuffled_perturbations)

n_total = len(shuffled_perturbations)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_perturbations = set(shuffled_perturbations[:n_train].tolist())
val_perturbations = set(shuffled_perturbations[n_train : n_train + n_val].tolist())
test_perturbations = set(shuffled_perturbations[n_train + n_val :].tolist())

print(f"Total perturbations from metadata: {n_total:,}")
print(f"Train perturbations: {len(train_perturbations):,}")
print(f"Val perturbations: {len(val_perturbations):,}")
print(f"Test perturbations: {len(test_perturbations):,}")

# Perturbations present in the currently streamed subset.
observed_perturbations = {
    perturbation
    for perturbation_map in data_by_cell_and_drug.values()
    for perturbation in perturbation_map.keys()
}
print(f"Perturbations observed in streamed subset: {len(observed_perturbations):,}")

Resolving data files:   0%|          | 0/3388 [00:00<?, ?it/s]

Total perturbations from metadata: 380
Train perturbations: 304
Val perturbations: 38
Test perturbations: 38
Perturbations observed in streamed subset: 95


In [5]:
class PerturbationDataset(Dataset):
    def __init__(
        self,
        data_by_cell_and_drug,
        allowed_perturbations,
        cell_line_to_idx,
        perturbation_to_idx,
    ):
        self.data_by_cell_and_drug = data_by_cell_and_drug
        self.allowed_perturbations = set(allowed_perturbations)
        self.cell_line_to_idx = cell_line_to_idx
        self.perturbation_to_idx = perturbation_to_idx

        self.samples = []
        for cell_line_id, perturbation_map in self.data_by_cell_and_drug.items():
            for perturbation, pair_adata in perturbation_map.items():
                if perturbation not in self.allowed_perturbations:
                    continue
                for row_idx in range(pair_adata.n_obs):
                    self.samples.append((cell_line_id, perturbation, row_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cell_line_id, perturbation, row_idx = self.samples[idx]
        pair_adata = self.data_by_cell_and_drug[cell_line_id][perturbation]

        sparse_row = pair_adata.X[row_idx]
        if hasattr(sparse_row, "toarray"):
            expression_np = sparse_row.toarray().ravel().astype(np.float32, copy=False)
        else:
            expression_np = np.asarray(sparse_row, dtype=np.float32).ravel()

        return {
            "expression": torch.from_numpy(expression_np),
            "cell_line_idx": torch.tensor(self.cell_line_to_idx[cell_line_id], dtype=torch.long),
            "perturbation_idx": torch.tensor(self.perturbation_to_idx[perturbation], dtype=torch.long),
            "cell_line_id": cell_line_id,
            "perturbation": perturbation,
        }


all_cell_lines = sorted(data_by_cell_and_drug.keys())
cell_line_to_idx = {cell_line_id: i for i, cell_line_id in enumerate(all_cell_lines)}
perturbation_to_idx = {
    perturbation: i for i, perturbation in enumerate(sorted(all_perturbations))
}

# Restrict each split to perturbations that are present in the streamed subset.
train_perturbations_observed = train_perturbations & observed_perturbations
val_perturbations_observed = val_perturbations & observed_perturbations
test_perturbations_observed = test_perturbations & observed_perturbations

train_ds = PerturbationDataset(
    data_by_cell_and_drug,
    train_perturbations_observed,
    cell_line_to_idx,
    perturbation_to_idx,
)
val_ds = PerturbationDataset(
    data_by_cell_and_drug,
    val_perturbations_observed,
    cell_line_to_idx,
    perturbation_to_idx,
)
test_ds = PerturbationDataset(
    data_by_cell_and_drug,
    test_perturbations_observed,
    cell_line_to_idx,
    perturbation_to_idx,
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [6]:
def count_cells_in_split(data_by_cell_and_drug, perturbation_set):
    total_cells = 0
    for perturbation_map in data_by_cell_and_drug.values():
        for perturbation, pair_adata in perturbation_map.items():
            if perturbation in perturbation_set:
                total_cells += int(pair_adata.n_obs)
    return total_cells


print("=== Perturbation split (global metadata) ===")
print(f"Train perturbations: {len(train_perturbations):,}")
print(f"Val perturbations:   {len(val_perturbations):,}")
print(f"Test perturbations:  {len(test_perturbations):,}")

print("\n=== Perturbations represented in streamed subset ===")
print(f"Train perturbations observed: {len(train_perturbations_observed):,}")
print(f"Val perturbations observed:   {len(val_perturbations_observed):,}")
print(f"Test perturbations observed:  {len(test_perturbations_observed):,}")

print("\n=== Cell counts represented in streamed subset ===")
print(f"Train cells: {count_cells_in_split(data_by_cell_and_drug, train_perturbations_observed):,}")
print(f"Val cells:   {count_cells_in_split(data_by_cell_and_drug, val_perturbations_observed):,}")
print(f"Test cells:  {count_cells_in_split(data_by_cell_and_drug, test_perturbations_observed):,}")

print("\n=== Dataset lengths ===")
print(f"len(train_ds): {len(train_ds):,}")
print(f"len(val_ds):   {len(val_ds):,}")
print(f"len(test_ds):  {len(test_ds):,}")

if len(train_ds) == 0:
    print("\nTrain dataset is empty. Increase SAMPLE_SIZE or set SAMPLE_SIZE=None.")
else:
    train_batch = next(iter(train_loader))
    print("\n=== One train batch ===")
    print("expression shape:", tuple(train_batch["expression"].shape))
    print("cell_line_idx shape:", tuple(train_batch["cell_line_idx"].shape))
    print("perturbation_idx shape:", tuple(train_batch["perturbation_idx"].shape))
    print("example cell_line_ids:", train_batch["cell_line_id"][:5])
    print("example perturbations:", train_batch["perturbation"][:5])

=== Perturbation split (global metadata) ===
Train perturbations: 304
Val perturbations:   38
Test perturbations:  38

=== Perturbations represented in streamed subset ===
Train perturbations observed: 68
Val perturbations observed:   16
Test perturbations observed:  10

=== Cell counts represented in streamed subset ===
Train cells: 142,734
Val cells:   28,963
Test cells:  25,129

=== Dataset lengths ===
len(train_ds): 142,734
len(val_ds):   28,963
len(test_ds):  25,129

=== One train batch ===
expression shape: (256, 0)
cell_line_idx shape: (256,)
perturbation_idx shape: (256,)
example cell_line_ids: ['CVCL_0332', 'CVCL_0546', 'CVCL_0397', 'CVCL_0292', 'CVCL_0131']
example perturbations: ['Larotrectinib', 'Norepinephrine (hydrochloride)', 'Bergenin', 'Diammonium Glycyrrhizinate', 'Raltitrexed']
